### Models

#### Invoking Model

In [ ]:
from langchain.chat_models import init_chat_model
# from dotenv import load_dotenv
# load_dotenv()
# import os

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    model="ollama:deepseek-r1:1.5b",
    temperature = 0.5,
    max_tokens = 2500,
    max_retries = 2,
    timeout= 30,
)

response = model.invoke("What is a parrot ?")
print(response.content)

#### Streaming

In [ ]:
stream = model.stream("Which is the most powerful animal in entire world")
for chunk in stream:
    print(chunk.text, end="|", flush=True)

#### Batching

##### With Entire Output

In [ ]:
batch_response = model.batch([
    "What is purpose of 2 kidneys in human body?",
    "Why elephants can not fly?",
    "Why mens have beard but women have not ?"
])

for response in batch_response:
    print(response.text)

##### When Completed

In [ ]:
batch_response = model.batch_as_completed([
    "What is purpose of 2 kidneys in human body?",
    "Why elephants can not fly?",
    "Why mens have beard but women have not ?"
])

print(batch_response)

### Tools

In [ ]:
from langchain.tools import tool

@tool
def getResturants(location: str) -> str:
    """Get all the resturants location details and their avarage ratings"""
    return f"Searching Resturants {location}"
@tool
def getSquareOfNum(num: int) -> int:
    "Get the squres of given number"
    return f"Square of number is {num * num}"

model_with_tools = model.bind_tools(tools=[getResturants, getSquareOfNum])
response = model_with_tools.invoke(["what is square of number 5", "Famous resturents in bengaluru and their ratings"])

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

##### Tool Output to Model

In [36]:
my_tool = [getResturants, getSquareOfNum]
tool_by_name = {tool.name: tool for tool in my_tool}
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool = tool_by_name[tool_call['name']]
    print(f"Tool: {tool}")